In [6]:
CSV_PATH = "../data/hospital/polluted/aggregationofpollution.csv"
OUTPUT_DIR = "./results"

In [14]:
import os
import re
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# ============================================================
# Configuration
# ============================================================

FD_HOLD_THRESHOLD = 0.95
DROP_MOJIBAKE = True
FILTER_GPDEP_GT = None
INCLUDE_RELAXED_PROXIES = False
GENERATE_GROUPWISE = False
MIN_OBS = 3

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# Load data
# ============================================================
df = pd.read_csv(CSV_PATH)

ERROR_TYPE_MAP = {
    "Permutate_dash": "Permutate",
    "Permutate_space": "Permutate",
    "Permutate_underscore": "Permutate",
}

df["error_type_table"] = df["error_type"].replace(ERROR_TYPE_MAP)

if DROP_MOJIBAKE:
    df = df[df["error_type_table"] != "Mojibake"].copy()

if FILTER_GPDEP_GT is not None:
    df = df[df["gpdep"] > FILTER_GPDEP_GT].copy()

# ============================================================
# Helpers
# ============================================================
def safe_weighted_average(values: pd.Series, weights: pd.Series) -> float:
    tmp = pd.concat([values, weights], axis=1).dropna()
    if tmp.empty:
        return np.nan

    v = tmp.iloc[:, 0].astype(float).to_numpy()
    w = tmp.iloc[:, 1].astype(float).clip(lower=0).to_numpy()

    if np.isclose(w.sum(), 0):
        return np.nan

    return float(np.average(v, weights=w))


def safe_weighted_binary_score(binary_values: pd.Series, weights: pd.Series) -> float:
    tmp = pd.concat([binary_values, weights], axis=1).dropna()
    if tmp.empty:
        return np.nan

    b = tmp.iloc[:, 0].astype(float).to_numpy()
    w = tmp.iloc[:, 1].astype(float).clip(lower=0).to_numpy()

    if np.isclose(w.sum(), 0):
        return np.nan

    return float(np.sum(w * b) / np.sum(w))


def safe_pearson(x: pd.Series, y: pd.Series, min_obs: int = 3):
    tmp = pd.concat([x, y], axis=1).dropna()
    if len(tmp) < min_obs:
        return np.nan, np.nan
    if tmp.iloc[:, 0].nunique() < 2:
        return np.nan, np.nan
    if tmp.iloc[:, 1].nunique() < 2:
        return np.nan, np.nan
    r, p = pearsonr(tmp.iloc[:, 0], tmp.iloc[:, 1])
    return float(r), float(p)


def corr_and_p_matrix(data: pd.DataFrame, metric_cols):
    metric_cols = [c for c in metric_cols if c in data.columns]

    r_mat = pd.DataFrame(np.nan, index=metric_cols, columns=metric_cols)
    p_mat = pd.DataFrame(np.nan, index=metric_cols, columns=metric_cols)

    for c1 in metric_cols:
        for c2 in metric_cols:
            if c1 == c2:
                r_mat.loc[c1, c2] = 1.0
                p_mat.loc[c1, c2] = 0.0
            else:
                r, p = safe_pearson(data[c1], data[c2], min_obs=MIN_OBS)
                r_mat.loc[c1, c2] = r
                p_mat.loc[c1, c2] = p

    return r_mat, p_mat


def sanitize_filename(text: str) -> str:
    text = str(text)
    text = re.sub(r"[^\w\-_. ]", "_", text)
    text = re.sub(r"\s+", "_", text)
    return text


def nice_metric_name(metric: str) -> str:
    rename = {
        "gpdep_mean": "Mean gpdep",
        "partial_value_mean": "Mean partial value",
        "fd_hold_pct": "Share of FDs holding",
        "fd_hold_weighted": "Share of Weighted FDs holding",
        "qw_lin": "Qw-lin",
        "qw_bin": "C_{pFD}",
        "cordts_pipino": "Cordts/Pipino",
        "heinrich_product_exact": "Heinrich et al.",
        "hinrichs_exact": "Hinrichs",
    }
    return rename.get(metric, metric)


def relabel_matrix(mat: pd.DataFrame) -> pd.DataFrame:
    out = mat.copy()
    out.index = [nice_metric_name(x) for x in out.index]
    out.columns = [nice_metric_name(x) for x in out.columns]
    return out


def plot_matrix_heatmap(
    mat: pd.DataFrame,
    title: str,
    out_prefix: str,
    value_format: str = ".3f",
    cmap: str = "coolwarm",
    vmin=None,
    vmax=None
):
    if mat.empty:
        return

    plot_df = relabel_matrix(mat)
    values = plot_df.to_numpy(dtype=float)

    n_rows, n_cols = values.shape
    fig_width = max(6, 1.4 * n_cols)
    fig_height = max(5, 1.1 * n_rows)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    im = ax.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")

    ax.set_xticks(np.arange(n_cols))
    ax.set_yticks(np.arange(n_rows))
    ax.set_xticklabels(plot_df.columns, rotation=45, ha="right")
    ax.set_yticklabels(plot_df.index)
    ax.set_title(title, pad=14)

    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=1)
    ax.tick_params(which="minor", bottom=False, left=False)

    for i in range(n_rows):
        for j in range(n_cols):
            val = values[i, j]
            txt = "--" if pd.isna(val) else format(val, value_format)
            ax.text(j, i, txt, ha="center", va="center", color="black", fontsize=10)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Value", rotation=270, labelpad=15)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{out_prefix}.pdf", bbox_inches="tight")
    plt.close(fig)


def build_groupwise_matrices(data: pd.DataFrame, group_cols, metric_cols):
    out = {}

    for group_key, group_df in data.groupby(group_cols, dropna=False):
        if len(group_df) < MIN_OBS:
            continue

        local_cols = []
        for c in metric_cols:
            if c not in group_df.columns:
                continue
            if group_df[c].notna().sum() < MIN_OBS:
                continue
            if group_df[c].nunique(dropna=True) < 2:
                continue
            local_cols.append(c)

        if len(local_cols) < 2:
            continue

        r_mat, p_mat = corr_and_p_matrix(group_df, local_cols)
        out[group_key] = {"n": len(group_df), "r": r_mat, "p": p_mat}

    return out

# ============================================================
# Build one row per corrupted dataset variant
# ============================================================
VARIANT_KEYS = [
    "dataset_name",
    "attribute_name",
    "error_mechanic",
    "error_type_table",
    "error_rate",
    "data_file",
]

rows = []
n_attributes = df["attribute_name"].nunique()
for key_vals, g in df.groupby(VARIANT_KEYS, dropna=False):
    row = dict(zip(VARIANT_KEYS, key_vals))

    row["n_fds"] = len(g)

    row["gpdep_mean"] = g["gpdep"].mean()
    row["partial_value_mean"] = g["partial_value"].mean()

    # Binary hold indicator per FD
    holds = (g["partial_value"] >= FD_HOLD_THRESHOLD).astype(int)
    strict_violation = 1 - holds
    
    # Unweighted share of holding FDs
    row["fd_hold_pct"] = holds.mean()
    row["fd_hold_weighted"] = row["gpdep_mean"] * row["fd_hold_pct"]
    
    # Violated FD share if you still want it elsewhere
    row["fd_violated_pct"] = 1.0 - row["fd_hold_pct"]

    # New weighted binary score
    row["qw_bin"] = safe_weighted_binary_score(holds, g["gpdep"])

    # Strictly computable related-work metrics
    row["cordts_pipino"] = max(0.0, 1.0 - (strict_violation.sum() / n_attributes))
    
    row["heinrich_product_exact"] = (1 - strict_violation).prod()
    row["hinrichs_exact"] = 1.0 / (strict_violation.sum() + 1.0)

    gpdep_weight = g["gpdep"].astype(float).clip(lower=0).fillna(0.0)
    weighted_strict_violation = strict_violation.astype(float) * gpdep_weight
    
    weighted_violation_sum = weighted_strict_violation.sum()

    rows.append(row)

variant_df = pd.DataFrame(rows)
variant_df.to_csv(os.path.join(OUTPUT_DIR, "variant_level_metrics.csv"), index=False)

# ============================================================
# Select metrics for the matrix
# ============================================================
metric_cols = [
    "gpdep_mean",
    "partial_value_mean",
    "fd_hold_pct",
    "fd_hold_weighted",
    "qw_bin",
    "heinrich_product_exact",
    "hinrichs_exact",
    "cordts_pipino",
]

usable_metric_cols = []
for c in metric_cols:
    if c not in variant_df.columns:
        continue
    if not variant_df[c].notna().any():
        continue
    if variant_df[c].nunique(dropna=True) < 2:
        print(f"Skipping constant column: {c}")
        continue
    usable_metric_cols.append(c)

# ============================================================
# Overall correlation matrices
# ============================================================
corr_r, corr_p = corr_and_p_matrix(variant_df, usable_metric_cols)

corr_r.to_csv(os.path.join(OUTPUT_DIR, "corr_matrix_r.csv"))
corr_p.to_csv(os.path.join(OUTPUT_DIR, "corr_matrix_p.csv"))

plot_matrix_heatmap(
    mat=corr_r,
    title="Pearson correlation matrix between dataset-level metrics",
    out_prefix=os.path.join(OUTPUT_DIR, "corr_matrix_r"),
    value_format=".3f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plot_matrix_heatmap(
    mat=corr_p,
    title="P-value matrix for Pearson correlations",
    out_prefix=os.path.join(OUTPUT_DIR, "corr_matrix_p"),
    value_format=".4f",
    cmap="viridis_r",
    vmin=0,
    vmax=1
)

# ============================================================
# Optional: group-wise figures
# ============================================================
if GENERATE_GROUPWISE:
    group_cols = ["dataset_name", "error_mechanic", "error_type_table"]
    groupwise = build_groupwise_matrices(variant_df, group_cols, usable_metric_cols)

    for key, mats in groupwise.items():
        key_str = "__".join(map(str, key))
        key_safe = sanitize_filename(key_str)

        mats["r"].to_csv(os.path.join(OUTPUT_DIR, f"corr_matrix_r__{key_safe}.csv"))
        mats["p"].to_csv(os.path.join(OUTPUT_DIR, f"corr_matrix_p__{key_safe}.csv"))

        plot_matrix_heatmap(
            mat=mats["r"],
            title=f"Pearson correlation matrix ({key_str}, n={mats['n']})",
            out_prefix=os.path.join(OUTPUT_DIR, f"corr_matrix_r__{key_safe}"),
            value_format=".3f",
            cmap="coolwarm",
            vmin=-1,
            vmax=1
        )

        plot_matrix_heatmap(
            mat=mats["p"],
            title=f"P-value matrix ({key_str}, n={mats['n']})",
            out_prefix=os.path.join(OUTPUT_DIR, f"corr_matrix_p__{key_safe}"),
            value_format=".4f",
            cmap="viridis_r",
            vmin=0,
            vmax=1
        )

print("Done.")
print(f"Saved outputs to: {OUTPUT_DIR}")

Done.
Saved outputs to: ./results
